In [10]:
import pandas as pd
import sqlite3

In [11]:
conn = sqlite3.connect("../data/checking-logs.sqlite")
schema = pd.read_sql("PRAGMA table_info(pageviews)", conn)
print(schema)

   cid      name       type  notnull dflt_value  pk
0    0     index    INTEGER        0       None   0
1    1       uid       TEXT        0       None   0
2    2  datetime  TIMESTAMP        0       None   0


In [12]:
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS datamart;")
cursor.execute("""
    CREATE TABLE datamart AS
    WITH first_views AS (
        SELECT uid, MIN(datetime) AS first_view_ts
        FROM pageviews
        WHERE uid LIKE 'user_%'
        GROUP BY uid
    )
    SELECT
        c.uid,
        c.labname,
        MIN(c.timestamp) AS first_commit_ts,
        fv.first_view_ts
    FROM checker c
    LEFT JOIN first_views fv ON c.uid = fv.uid
    WHERE
        c.status = 'ready'
        AND c.numTrials = 1
        AND c.labname IN ('laba04', 'laba04s', 'laba05', 'laba06', 'laba06s', 'project1')
        AND c.uid LIKE 'user_%'
    GROUP BY c.uid, c.labname
""")
conn.commit()

In [13]:
datamart = pd.read_sql("SELECT * FROM datamart;", conn)

datamart['first_commit_ts'] = pd.to_datetime(datamart['first_commit_ts'])
datamart['first_view_ts'] = pd.to_datetime(datamart['first_view_ts'])

In [14]:
datamart.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              140 non-null    object        
 1   labname          140 non-null    object        
 2   first_commit_ts  140 non-null    datetime64[ns]
 3   first_view_ts    59 non-null     datetime64[ns]
dtypes: datetime64[ns](2), object(2)
memory usage: 4.5+ KB


In [15]:
test = datamart[datamart['first_view_ts'].notnull()]
control = datamart[datamart['first_view_ts'].isnull()]

avg = test['first_view_ts'].mean()
print(avg)

control.loc[:, 'first_view_ts'] = avg

2020-04-27 00:40:05.761783552


In [16]:
test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 59 entries, 0 to 114
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              59 non-null     object        
 1   labname          59 non-null     object        
 2   first_commit_ts  59 non-null     datetime64[ns]
 3   first_view_ts    59 non-null     datetime64[ns]
dtypes: datetime64[ns](2), object(2)
memory usage: 2.3+ KB


In [17]:
control.info()

<class 'pandas.core.frame.DataFrame'>
Index: 81 entries, 12 to 139
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              81 non-null     object        
 1   labname          81 non-null     object        
 2   first_commit_ts  81 non-null     datetime64[ns]
 3   first_view_ts    81 non-null     datetime64[ns]
dtypes: datetime64[ns](2), object(2)
memory usage: 3.2+ KB


In [18]:
test.to_sql('test', conn, if_exists='replace', index=False)
control.to_sql('control', conn, if_exists='replace', index=False)

conn.close()